In [1]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent,middleware
from langchain.tools import tool
from langchain.agents.middleware import (dynamic_prompt,wrap_model_call)
import os

In [2]:
# 低级模型

base_model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V4-Flash",
    api_key=os.getenv("GUIJI_API_KEY"),
    base_url=os.getenv("GUIJI_BASE_URL"),
    temperature=0
)
# 高级模型
advanced_model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V4-Flash",
    api_key=os.getenv("GUIJI_API_KEY"),
    base_url=os.getenv("GUIJI_BASE_URL"),
    temperature=0
)

In [3]:
@tool
def get_location()->str:
    """获取当前的地理位置"""
    return "当前位置是北京"

@tool
def get_weather(city:str)->str:
    """获取指定位置的天气"""
    return f"{city}当前的天气是晴天"

In [8]:
from typing import TypedDict
from pydantic import BaseModel

class AgentContext(BaseModel):
    query_type:str
    uid:int


In [14]:
# 定义中间件
from langchain.agents.middleware import ModelRequest, ModelResponse

# 动态模型中间件    
@wrap_model_call
def dynamic_model_selection(request:ModelRequest,handler)->ModelResponse:
    print("request:",request)
    # 判断消息条数
    message_count = len(request.state['messages'])
    if message_count < 3:
        model = base_model
    else:
        model = advanced_model
        print("模型切换辣")
    return handler(request.override(model=model))

# 动态提示词中间件
@dynamic_prompt
def dynamic_support_prompt(request:ModelRequest)->str:
    query_type = request.runtime.context.query_type
    query_uid = request.runtime.context.uid
    print(query_type,query_uid)
    return " "

In [15]:
agent = create_agent(
    model=base_model,
    tools=[get_location,get_weather],
    middleware=[dynamic_model_selection,dynamic_support_prompt],
    context_schema=AgentContext 
)

In [16]:
response = agent.invoke(
    {
        "messages":[
            {"role":"system","content":"你是一个天气助手"},
            {"role":"user","content":"我现在位置的天气如何"},
        ]
    },
    context={"query_type":"vip","uid":666}
)

request: ModelRequest(model=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000002EB42AC5410>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002EB434A5610>, root_client=<openai.OpenAI object at 0x000002EB42A0A290>, root_async_client=<openai.AsyncOpenAI object at 0x000002EB434A4A90>, model_name='deepseek-ai/DeepSeek-V4-Flash', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.siliconflow.cn/v1', openai_proxy=None, stream_chunk_timeout=120.0), messages=[SystemMessage(content='你是一个天气助手', additional_kwargs={}, response_metadata={}, id='e81b50e2-7fe7-4b00-b0ee-8286540cee8f'), HumanMessage(content='我现在位置的天气如何', additional_kwargs={}, response_metadata={}, id='3e5f61d0-77a3-404d-9f17-4a99a2c9c497')], system_message=None, tool_choi

In [16]:
for msg in response["messages"]:
    msg.pretty_print()

================================ System Message ================================

你是一个天气助手
================================ Human Message =================================

我现在位置的天气如何
================================== Ai Message ==================================
Tool Calls:
  get_location (019fc1b45a349850445384ca05c3c242)
 Call ID: 019fc1b45a349850445384ca05c3c242
  Args:
================================= Tool Message =================================
Name: get_location

当前位置是北京
================================== Ai Message ==================================
Tool Calls:
  get_weather (019fc1b46fa14936f0b3cbfa0311b58c)
 Call ID: 019fc1b46fa14936f0b3cbfa0311b58c
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京当前的天气是晴天
================================== Ai Message ==================================

你当前的位置是**北京**，天气为**晴天**。☀️

请问还有其他需要帮忙的吗？
